# CLARA: Comprehension and Literacy Assessment for Readability Analysis

Welcome to the CLARA evaluation notebook! CLARA is a systematic platform designed to evaluate reading comprehension in smaller language models and perform detailed error analysis to diagnose why they fail.

This notebook leverages **Gemini Flash** (via API) to generate a fictitious reading passage and a set of multiple-choice questions split among three difficulty levels. We then run and compare models from the **GPT-2** family to observe how model scale impacts adherence to instructions, formatting stability, and overall comprehension accuracy. Results are persistently cached in Google Drive (or local workspace fallback).

### Cell 1: Environment Setup
**Description:**
This cell installs the required third-party libraries for the CLARA platform. These include `google-generativeai` (for accessing Gemini Flash), Hugging Face `transformers` and `accelerate` (for running GPT-2 models locally), and visualization libraries (`matplotlib`, `pandas`). Additionally, it attempts to mount Google Drive for persistent caching. If run outside of Google Colab, Google Drive mounting will gracefully fail or be skipped, and the platform will fall back to local caching.

In [ ]:
import os
import sys

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Google Colab: {IN_COLAB}")

# Install necessary libraries if in Colab or if they are missing
if IN_COLAB:
    print("Installing packages...")
    !pip install -q google-generativeai transformers accelerate pandas matplotlib torch
else:
    print("In local environment. Ensure you have installed: google-generativeai, transformers, pandas, matplotlib, torch, accelerate")

# Mount Google Drive if in Colab
if IN_COLAB:
    from google.colab import drive
    try:
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
    except Exception as e:
        print("Could not mount Google Drive. Local caching will be used instead.")
        print(f"Error: {e}")

### Cell 2: Configuration & Parameters
**Description:**
This cell contains all the configurable parameters for the CLARA evaluation framework. You can specify your Gemini API key, select which Gemini model to use for story and question generation, choose which GPT-2 model sizes to run for the ablation study, configure the number of sampling trials per question (between 10 and 50), set the temperature, and define the caching directories.

In [ ]:
import os

# --- Gemini Configuration ---
# Provide your Gemini API key here or set it in your environment / Colab Secrets
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")
GEMINI_MODEL = "gemini-2.5-flash"  # Fictitious story and question generator

# --- GPT-2 Ablation Configuration ---
# Choose from: 'gpt2' (small), 'gpt2-medium', 'gpt2-large', 'gpt2-xl'
# We will compare performance across these models.
GPT2_MODELS_TO_EVALUATE = ["gpt2", "gpt2-medium"]  # Customize as needed for your runtime capacity

# --- Evaluation Parameters ---
RUN_COUNT = 20  # Number of generation trials per question (typically 10-50)
TEMPERATURE = 0.7  # Temperature for GPT-2 answer sampling
MAX_NEW_TOKENS = 10  # Max tokens to generate to find the answer

# --- Caching Paths ---
# If Google Drive is mounted, we store the files there. Otherwise, we fallback to a local cache.
GOOGLE_DRIVE_DIR = "/content/drive/MyDrive/clara_cache"
LOCAL_DIR = "./clara_cache"

print("Configurations loaded successfully.")
print(f"Gemini Model: {GEMINI_MODEL}")
print(f"GPT-2 Models for Ablation: {GPT2_MODELS_TO_EVALUATE}")
print(f"Runs per question: {RUN_COUNT}")

### Cell 3: Google Drive Caching Utilities
**Description:**
This cell implements robust caching utilities. These functions determine whether Google Drive is mounted and accessible. If so, caching is done in `/content/drive/MyDrive/clara_cache/`. If Google Drive is unavailable, the system automatically falls back to `./clara_cache/` in the local workspace. This preserves experiment outputs and prevents redundant API calls or heavy model inferences if the notebook is re-run.

In [ ]:
import json
import os

def get_cache_directory():
    """Returns the mounted Google Drive cache path if available, else local path."""
    if os.path.exists("/content/drive/MyDrive"):
        os.makedirs(GOOGLE_DRIVE_DIR, exist_ok=True)
        return GOOGLE_DRIVE_DIR
    else:
        os.makedirs(LOCAL_DIR, exist_ok=True)
        return LOCAL_DIR

def save_to_cache(filename, data):
    """Saves data (dict/list) as JSON to the active cache directory."""
    cache_dir = get_cache_directory()
    filepath = os.path.join(cache_dir, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved cache to: {filepath}")

def load_from_cache(filename):
    """Loads JSON data from the active cache directory if it exists, else returns None."""
    cache_dir = get_cache_directory()
    filepath = os.path.join(cache_dir, filename)
    if os.path.exists(filepath):
        print(f"Loading cache from: {filepath}")
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

print(f"Active cache directory is: {get_cache_directory()}")

### Cell 4: Gemini Story & Question Generator
**Description:**
This cell uses the Gemini API (via the `google-generativeai` SDK) to generate a unique, fictitious story and 9 multiple-choice comprehension questions split equally across three difficulty levels (3 Easy, 3 Medium, and 3 Hard). To guarantee structured output, we supply a detailed prompt and ask for JSON output conforming to a specific schema. If a cached story exists in Drive or the local workspace, it loads it instead of calling the API.

In [ ]:
import google.generativeai as genai
import json

# Setup the Gemini API key
if GEMINI_API_KEY and GEMINI_API_KEY != "YOUR_GEMINI_API_KEY":
    genai.configure(api_key=GEMINI_API_KEY)
else:
    # Attempt to load from environment variable or Colab Secrets
    try:
        from google.colab import userdata
        key = userdata.get('GEMINI_API_KEY')
        genai.configure(api_key=key)
        print("Loaded Gemini API key from Colab Secrets.")
    except Exception:
        print("WARNING: Gemini API Key is not set! Set GEMINI_API_KEY in Cell 2 or as a secret.")

STORY_CACHE_FILE = "story_questions.json"

def generate_story_and_questions():
    """Generates a fictitious story and MC questions split by difficulty using Gemini."""
    prompt = """
    Generate a completely fictitious, short story (around 300-500 words).
    Based on this story, generate exactly 9 multiple-choice reading comprehension questions.
    The questions must be divided equally among three difficulty levels:
    - 3 Easy: Literal recall questions, directly answered in a single sentence of the text.
    - 3 Medium: Inference/integration questions, requiring combining information from multiple parts of the text.
    - 3 Hard: Critical thinking/evaluation questions, requiring understanding tone, theme, or complex implications.

    For each question, provide 4 options (A, B, C, D) and specify the single correct_answer ("A", "B", "C", or "D").
    Also provide a brief explanation of why that answer is correct.

    Your output MUST be a valid JSON object matching this schema:
    {
      "story": "The text of the story here...",
      "questions": [
        {
          "id": 1,
          "difficulty": "Easy",
          "question": "The text of the question?",
          "options": {
            "A": "Option A text",
            "B": "Option B text",
            "C": "Option C text",
            "D": "Option D text"
          },
          "correct_answer": "A",
          "explanation": "Brief explanation..."
        },
        ...
      ]
    }
    Ensure the JSON is well-formed, valid, and contains exactly 9 questions (3 of each difficulty). Do not wrap the JSON in markdown code blocks like ```json ... ```. Just return raw JSON.
    """
    
    print(f"Calling Gemini ({GEMINI_MODEL}) to generate story and questions...")
    
    # Configure generation to request JSON output
    model = genai.GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(
        prompt,
        generation_config={"response_mime_type": "application/json"}
    )
    
    content = response.text.strip()
    # Parse and validate JSON
    data = json.loads(content)
    return data

# Main execution for Cell 4: Load from cache or generate
cached_data = load_from_cache(STORY_CACHE_FILE)
if cached_data:
    story_data = cached_data
    print("Successfully loaded story and questions from cache.")
else:
    try:
        story_data = generate_story_and_questions()
        save_to_cache(STORY_CACHE_FILE, story_data)
        print("Successfully generated and cached story and questions.")
    except Exception as e:
        print(f"Error generating story: {e}")
        # Create a dummy story dataset in case of API failure so the notebook remains runnable
        print("Creating a fallback/dummy story dataset for demonstration.")
        story_data = {
            "story": "The Clockmaker of Veridia. Long ago in the clockwork city of Veridia, a quiet clockmaker named Alistair crafted a mechanical bird that sang with the rising sun. Unlike other automatons, this bird possessed a small blue gemstone at its core. Legend whispered that the gemstone could sense the truth of a person's heart. One day, a suspicious merchant tried to buy the bird for a chest of gold, but Alistair refused, stating that some creations are meant to bring joy, not wealth. That night, the gemstone vanished, and the bird remained silent.",
            "questions": [
                {
                    "id": 1,
                    "difficulty": "Easy",
                    "question": "What did Alistair craft?",
                    "options": {
                        "A": "A mechanical fish",
                        "B": "A clockwork tower",
                        "C": "A mechanical bird",
                        "D": "A gold pocket watch"
                    },
                    "correct_answer": "C",
                    "explanation": "The text explicitly states Alistair crafted a mechanical bird."
                },
                {
                    "id": 2,
                    "difficulty": "Easy",
                    "question": "What is the name of the clockwork city?",
                    "options": {
                        "A": "Veridia",
                        "B": "Alistair",
                        "C": "Aurum",
                        "D": "Chronos"
                    },
                    "correct_answer": "A",
                    "explanation": "The text states the story takes place in the clockwork city of Veridia."
                },
                {
                    "id": 3,
                    "difficulty": "Easy",
                    "question": "What was at the core of the mechanical bird?",
                    "options": {
                        "A": "A clockwork spring",
                        "B": "A small blue gemstone",
                        "C": "A red ruby",
                        "D": "A gold gear"
                    },
                    "correct_answer": "B",
                    "explanation": "The text mentions a small blue gemstone at its core."
                },
                {
                    "id": 4,
                    "difficulty": "Medium",
                    "question": "Why did Alistair refuse to sell the bird to the merchant?",
                    "options": {
                        "A": "Because the merchant offered too little gold",
                        "B": "Because the bird was broken",
                        "C": "Because some creations are meant to bring joy, not wealth",
                        "D": "Because the bird belonged to the mayor"
                    },
                    "correct_answer": "C",
                    "explanation": "Alistair stated that some creations are meant to bring joy, not wealth, refusing the gold."
                },
                {
                    "id": 5,
                    "difficulty": "Medium",
                    "question": "What consequence followed the disappearance of the gemstone?",
                    "options": {
                        "A": "The city went into chaos",
                        "B": "The bird remained silent",
                        "C": "Alistair left Veridia",
                        "D": "The merchant got rich"
                    },
                    "correct_answer": "B",
                    "explanation": "The text states that once the gemstone vanished, the bird remained silent."
                },
                {
                    "id": 6,
                    "difficulty": "Medium",
                    "question": "What was rumored about the gemstone?",
                    "options": {
                        "A": "It could turn lead into gold",
                        "B": "It could sense the truth of a person's heart",
                        "C": "It had the power to stop time",
                        "D": "It belonged to a king"
                    },
                    "correct_answer": "B",
                    "explanation": "The story mentions that legend whispered the gemstone could sense the truth of a person's heart."
                },
                {
                    "id": 7,
                    "difficulty": "Hard",
                    "question": "Which theme is most prominent in the story?",
                    "options": {
                        "A": "The progress of modern industrialization",
                        "B": "The conflict between artistic integrity and commercial greed",
                        "C": "The inevitability of time passing",
                        "D": "The importance of legal systems"
                    },
                    "correct_answer": "B",
                    "explanation": "The interaction between the merchant offering gold and Alistair refusing emphasizes artistic/moral integrity vs commercialism."
                },
                {
                    "id": 8,
                    "difficulty": "Hard",
                    "question": "Based on the story's ending, what is the symbolic significance of the bird falling silent?",
                    "options": {
                        "A": "The bird was poorly designed",
                        "B": "The city lost its technological superiority",
                        "C": "Without its true, moral center (the gemstone), the bird's spirit or joy was gone",
                        "D": "Alistair forgot how to wind the bird"
                    },
                    "correct_answer": "C",
                    "explanation": "The gemstone represents heart and truth; its loss renders the joyful creation lifeless, underscoring the spiritual/moral loss."
                },
                {
                    "id": 9,
                    "difficulty": "Hard",
                    "question": "What does Alistair's refusal of a chest of gold reveal about his character?",
                    "options": {
                        "A": "He is wealthy already",
                        "B": "He values the intrinsic, non-monetary purpose of his art above financial gain",
                        "C": "He is stubborn and foolish",
                        "D": "He dislikes the merchant personally"
                    },
                    "correct_answer": "B",
                    "explanation": "Alistair is motivated by the joy his art brings, which cannot be bought by gold."
                }
            ]
        }
        save_to_cache(STORY_CACHE_FILE, story_data)

# Print a preview of the story and a question
print("\n--- STORY PREVIEW ---")
print(story_data["story"][:300] + "...")
print(f"Total Questions Generated: {len(story_data['questions'])}")

### Cell 5: GPT-2 Evaluator
**Description:**
This cell contains the evaluation engine for GPT-2 models. It downloads model weights from HuggingFace, constructs structured prompts matching the model's capabilities, runs multiple sampling trials (10 to 50 times per question) using temperature-controlled generation, and parses the outputs. GPT-2 models can be highly verbose or repetitive, so we implement robust parsing and validation. Any answer that cannot be resolved to a clean multiple-choice letter ("A", "B", "C", or "D") is discarded.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

def clean_and_parse_answer(new_text):
    """
    Parses the generated response from GPT-2.
    It expects the model to output something containing 'Answer: A' or just 'A'.
    """
    new_text = new_text.strip()
    
    # Check for direct answer patterns
    # Look for 'Answer: X' or 'Option: X' or '[X]' or 'X.' or just 'X' at the very beginning of generated text
    patterns = [
        r"(?:Answer|Option|Correct):\s*([A-D])",
        r"^\s*([A-D])\b",
        r"\[([A-D])\]",
        r"\b([A-D])\b"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, new_text, re.IGNORECASE)
        if match:
            parsed_letter = match.group(1).upper()
            return parsed_letter
            
    return None  # Discard if it doesn't meet the format

def evaluate_single_model(model_name, story_data, runs=20, temp=0.7):
    """
    Downloads model weights, runs evaluation, and returns raw results.
    We run 'runs' trials per question to record token choice distribution/stochastic behavior.
    """
    print(f"Loading weights for model: {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Set padding token to eos token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    print(f"Model loaded onto {device}.")
    
    story_text = story_data["story"]
    results = []
    
    for q in story_data["questions"]:
        q_id = q["id"]
        difficulty = q["difficulty"]
        question_text = q["question"]
        options = q["options"]
        correct_answer = q["correct_answer"]
        
        # Build options text
        options_str = "\n".join([f"{k}: {v}" for k, v in options.items()])
        
        # Structure the prompt for GPT-2
        prompt = f"Story:\n{story_text}\n\nQuestion: {question_text}\nOptions:\n{options_str}\n\nAnswer:"
        
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        valid_trials = 0
        total_trials = 0
        correct_count = 0
        trial_answers = []
        
        print(f"Running {runs} evaluation trials for Question {q_id} ({difficulty})...")
        
        # Loop-based allows clean logging of random samples
        for trial in range(runs):
            total_trials += 1
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=temp,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            # Extract and decode only the newly generated tokens
            input_len = inputs["input_ids"].shape[1]
            new_tokens = outputs[0][input_len:]
            new_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            parsed = clean_and_parse_answer(new_text)
            
            if parsed:
                valid_trials += 1
                is_correct = (parsed == correct_answer)
                if is_correct:
                    correct_count += 1
                trial_answers.append({
                    "trial": trial,
                    "raw_output": new_text,
                    "parsed_answer": parsed,
                    "is_correct": is_correct
                })
            else:
                trial_answers.append({
                    "trial": trial,
                    "raw_output": new_text,
                    "parsed_answer": None,
                    "is_correct": False
                })
                
        results.append({
            "question_id": q_id,
            "difficulty": difficulty,
            "question": question_text,
            "correct_answer": correct_answer,
            "total_trials": total_trials,
            "valid_trials": valid_trials,
            "correct_trials": correct_count,
            "accuracy": (correct_count / valid_trials) if valid_trials > 0 else 0.0,
            "trials_detail": trial_answers
        })
        
    return results

print("Evaluation engine ready.")

### Cell 6: Ablation Study Runner
**Description:**
This cell executes the ablation study across the specified GPT-2 models. It checks if the cache file for each model exists in Drive or local storage. If yes, it loads the cached result; otherwise, it triggers the evaluation, processes the results, and caches them immediately to avoid data loss.

In [ ]:
ablation_results = {}

for model_name in GPT2_MODELS_TO_EVALUATE:
    # Use clean file naming (replace slashes with underscores)
    safe_model_name = model_name.replace("/", "_")
    cache_file = f"gpt2_results_{safe_model_name}_{RUN_COUNT}.json"
    
    cached_res = load_from_cache(cache_file)
    if cached_res:
        print(f"Loaded evaluation results for {model_name} from cache.")
        ablation_results[model_name] = cached_res
    else:
        print(f"\n--- Evaluating Model: {model_name} ---")
        try:
            results = evaluate_single_model(model_name, story_data, runs=RUN_COUNT, temp=TEMPERATURE)
            save_to_cache(cache_file, results)
            ablation_results[model_name] = results
        except Exception as e:
            print(f"Error evaluating model {model_name}: {e}")
            
print("\nAll model evaluations completed/loaded.")

### Cell 7: Analysis & Error Diagnosis
**Description:**
This cell aggregates the raw simulation results, performs an in-depth error analysis, and computes performance stats. It outputs a clear summary of overall accuracy and segmented accuracies for each difficulty tier (Easy, Medium, Hard). It also lists specific questions where models frequently make errors, providing analytical insights into failure modes (e.g. invalid format rates, systematic bias towards specific letter keys).

In [ ]:
import pandas as pd

analysis_summary = []

for model_name, results in ablation_results.items():
    print(f"\n=================== ANALYSIS FOR {model_name.upper()} ===================")
    
    total_valid = 0
    total_correct = 0
    total_runs = 0
    
    difficulty_stats = {"Easy": {"correct": 0, "valid": 0}, "Medium": {"correct": 0, "valid": 0}, "Hard": {"correct": 0, "valid": 0}}
    
    failed_questions = []
    
    for res in results:
        diff = res["difficulty"]
        q_id = res["question_id"]
        valid = res["valid_trials"]
        correct = res["correct_trials"]
        total = res["total_trials"]
        acc = res["accuracy"]
        
        total_runs += total
        total_valid += valid
        total_correct += correct
        
        difficulty_stats[diff]["valid"] += valid
        difficulty_stats[diff]["correct"] += correct
        
        print(f"Q{q_id} ({diff}): Accuracy = {acc:.2%} ({correct}/{valid} valid trials, {total - valid} discarded)")
        
        # Track questions with less than 50% accuracy for failure analysis
        if acc < 0.5:
            failed_questions.append({
                "id": q_id,
                "question": res["question"],
                "difficulty": diff,
                "accuracy": acc,
                "discarded_ratio": (total - valid) / total if total > 0 else 0
            })
            
    overall_acc = total_correct / total_valid if total_valid > 0 else 0
    discard_rate = (total_runs - total_valid) / total_runs if total_runs > 0 else 0
    
    print(f"\n--- Summary for {model_name} ---")
    print(f"Overall Accuracy (Valid Trials Only): {overall_acc:.2%}")
    print(f"Format Discard Rate: {discard_rate:.2%}")
    
    model_stats = {
        "Model": model_name,
        "Overall Accuracy": overall_acc,
        "Discard Rate": discard_rate
    }
    
    for diff, stats in difficulty_stats.items():
        diff_acc = stats["correct"] / stats["valid"] if stats["valid"] > 0 else 0
        model_stats[f"{diff} Accuracy"] = diff_acc
        print(f"{diff} Level Accuracy: {diff_acc:.2%} ({stats['correct']}/{stats['valid']} trials)")
        
    analysis_summary.append(model_stats)
    
    if failed_questions:
        print("\n--- Failure Diagnostics ---")
        print("Questions where model had low accuracy (< 50%):")
        for fq in failed_questions:
            print(f"  - Q{fq['id']} [{fq['difficulty']}]: Accuracy = {fq['accuracy']:.1%}, Discarded Rate = {fq['discarded_ratio']:.1%}")
            print(f"    Text: \"{fq['question']}\"")
            
df_summary = pd.DataFrame(analysis_summary)
print("\n\n=== COMPARATIVE SUMMARY TABLE ===")
print(df_summary.to_string(index=False))

### Cell 8: Visualization & Insights
**Description:**
This cell visualizes the comparative performance of the GPT-2 model family across the three reading comprehension difficulty levels. It plots a multi-bar chart showing accuracy differences as model parameter size scales. Below the chart, we synthesize qualitative insights on why smaller causal language models struggle with comprehension, format alignment, and logical reasoning.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not df_summary.empty:
    models = df_summary["Model"].tolist()
    easy_accs = df_summary["Easy Accuracy"].tolist()
    med_accs = df_summary["Medium Accuracy"].tolist()
    hard_accs = df_summary["Hard Accuracy"].tolist()
    
    x = np.arange(len(models))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    rects1 = ax.bar(x - width, easy_accs, width, label='Easy (Literal Recall)', color='#4CAF50')
    rects2 = ax.bar(x, med_accs, width, label='Medium (Inference)', color='#FF9800')
    rects3 = ax.bar(x + width, hard_accs, width, label='Hard (Synthesis)', color='#F44336')
    
    ax.set_ylabel('Accuracy')
    ax.set_title('GPT-2 Ablation Study: Accuracy by Question Difficulty')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylim(0, 1.1)
    ax.legend(loc='upper right')
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add values on top of bars
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1%}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  # 3 points vertical offset
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=8)
            
    autolabel(rects1)
    autolabel(rects2)
    autolabel(rects3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(get_cache_directory(), "clara_ablation_chart.png"))
    plt.show()
    
    print("\n=== SYSTEMATIC INSIGHTS & CONCLUSIONS ===")
    print("1. SCALE VS ACCURACY: Larger models generally display lower discard rates and more consistent compliance with output formats.")
    print("2. DIFFICULTY CORRELATION: Easy (literal recall) questions typically see higher success. Medium and Hard questions require multi-hop inference or semantic abstraction, which exposes GPT-2's lack of instruction tuning.")
    print("3. FORMAT ATTENTIVENESS: Without reinforcement learning from human feedback (RLHF), GPT-2 family models often repeat prompt context or generate conversational filler rather than isolated answer keys. Discard rates serve as a proxy for raw schema adherence capabilities.")
else:
    print("No summary data available to plot.")